# Local Whisper in Colab — BrainHack 2026 Workshop

Hands-on part of module **4 (Data protection and local models)**.

Run the cells in order (`Shift+Enter`). Upload your own recording (~1 min) in the "1. Upload your own recording" section.
The model runs inside the Colab session — your audio goes to a Google machine, but **not** to external ASR provider APIs.


In [ ]:
# 0. Installation
!pip install -q "av<19" faster-whisper jiwer
import os


In [ ]:
# 1. Upload your own recording (~1 min, mp3/wav/m4a format)
from google.colab import files
uploaded = files.upload()
AUDIO = list(uploaded.keys())[0]
print('Uploaded:', AUDIO)


In [ ]:
# Play back the recording
from IPython.display import Audio, display
display(Audio(AUDIO))


## 2. Transcription (speech-to-text) — faster-whisper

The model runs locally within the Colab session. The result is a list of segments with timestamps and text.

We use the smaller **`small`** model — sufficient for presentation/demo purposes. Because of this, the results (transcription quality) may be less accurate than for larger models.


In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel('small', device='cpu', compute_type='int8')  # GPU: device='cuda', compute_type='float16'
segments, info = model.transcribe(AUDIO, language='pl')
segments = list(segments)

transcript = '\n'.join(s.text.strip() for s in segments)

for s in segments:
    print(f'[{s.start:6.1f}s -> {s.end:6.1f}s] {s.text.strip()}')

print('\n--- FULL TRANSCRIPT ---')
print(transcript)


## 3. Save to file (txt + SRT)

We save the transcript as plain text and as SRT subtitles, then download both files.


In [ ]:
from google.colab import files

base = os.path.splitext(AUDIO)[0]

# plain text
with open(f'{base}_transcript.txt', 'w', encoding='utf-8') as f:
    f.write(transcript)

# SRT (subtitles)
def srt_time(t):
    h, rem = divmod(t, 3600)
    m, s = divmod(rem, 60)
    return f'{int(h):02}:{int(m):02}:{int(s):02},{int((s % 1) * 1000):03}'

with open(f'{base}.srt', 'w', encoding='utf-8') as f:
    for i, s in enumerate(segments, 1):
        f.write(f'{i}\n{srt_time(s.start)} --> {srt_time(s.end)}\n{s.text.strip()}\n\n')

print('Saved:', f'{base}_transcript.txt', 'and', f'{base}.srt')
files.download(f'{base}_transcript.txt')
files.download(f'{base}.srt')


## 4. Optional: WER / CER metrics

WER/CER are computed against **your** manual transcript of **your** recording: paste it into the `reference` variable. If `reference` is empty, this cell skips the metrics.


In [ ]:
from jiwer import wer, cer

reference = ''  # paste your own manual transcript of your recording here


if reference.strip():
    print(f'WER: {wer(reference, transcript):.1%}')
    print(f'CER: {cer(reference, transcript):.1%}')
else:
    print('No reference provided — skipping WER/CER. Fill in the `reference` variable to compute the metrics.')


In [ ]:
# CER visualization in HTML — character-by-character differences
import difflib
from html import escape
from IPython.display import HTML, display

if reference.strip():
    RED = 'background:#680000;text-decoration:line-through'
    GREEN = 'background:#006b00'
    sm = difflib.SequenceMatcher(None, reference, transcript, autojunk=False)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == 'equal':
            parts.append(escape(reference[i1:i2]))
            continue
        if i2 > i1:
            parts.append(f'<span style="{RED}">{escape(reference[i1:i2])}</span>')
        if j2 > j1:
            parts.append(f'<span style="{GREEN}">{escape(transcript[j1:j2])}</span>')
    body = ''.join(parts)
    cer_score = cer(reference, transcript)
    display(HTML(f'''
    <div style="font-family:sans-serif;max-width:800px">
      <h3>CER: {cer_score:.1%}</h3>
      <p><span style="{RED}">red</span> = missing / error (reference) &nbsp;
         <span style="{GREEN}">green</span> = extra / wrong (transcript)</p>
      <div style="line-height:1.8;white-space:pre-wrap;border:1px solid #ccc;padding:10px;border-radius:6px">{body}</div>
    </div>'''))
else:
    print('No reference provided — skipping the CER visualization.')


## 5. Summary

Files produced: `*_transcript.txt` (text) and `*.srt` (subtitles). You can download them from the Colab files panel.

Privacy: the model runs inside the Colab session — your audio is not sent to external ASR APIs, but it is transferred to a Google machine (Colab). For a fully local setup, run Whisper on your own computer.
